In [ ]:
!pip install wfdb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 40.9 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.0 which is incompatible.
cudf-cu12 25.2.1 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.0 which is incompatible.
dask-cudf-cu12 25.2.2 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.0 which is incompatible.


In [ ]:
import pandas as pd
import os
import pickle

In [ ]:
!rm cargadatos.py

rm: cannot remove 'cargadatos.py': No such file or directory


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving cargadatos.py to cargadatos.py


In [ ]:
from cargadatos import proceso_interno_carga, carga_archivos_entorno

# Descarga de archivos de clasificación

En primer lugar vamos a crear una función que descargue desde PhysioNet los archivos con extensión '.csv' que recogen la clasificación de los ECG con los que vamos a trabajar. Esto lo necesitamos para posteriormente poder entrenar a nuestra red GAN.

In [ ]:
def descargar_archivo_clasificacion (ruta_csv, ruta_drive, nombre_archivo):

  """
  Esta función realiza la descarga de un archivo con extensión '.csv' desde la ruta 'ruta_csv' pasada como argumento en la ruta 'ruta_drive' bajo el nombre de 'nombre_archivo.
  Principalmente, sirve para descargar los archivos que encontramos en la base de datos de PhysioNet y que relacionan el nombre de cada uno de los registros con los que vamos a
  trabajar con su condición clínica, es decir, recoge la clasificación de estos.
  Es importante indicar el tipo de separador que va a utilizar nuestro archivo (coma en este caso) para evitar posibles problemas de lectura.
  Además, convierte este archivo con extensión '.csv' en un DataFram de Pandas, y lo devuelve como resultado, de forma que podamos trabajar con él en Python de una forma mucho
  más sencilla, y con mayor eficacia.

  Parámetros
  -------------
    - ruta_csv (str): Ruta de PhysioNet en la que se encuentra el archivo con extensión '.csv' que recoge la clasificación de los registros de ECG con los que vamos a trabajar.
      Ejemplo: "https://physionet.org/files/challenge-2017/1.0.0/REFERENCE-v0.csv"

    - ruta_drive (str): Ruta de Google Drive en la que se va a almacenar el archivo descargado desde la ruta 'ruta_csv', que correponde a la ruta PhysioNet.
      Ejemplo: "/content/drive/MyDrive/dataset_ECG/challenge2017/clasificacion"

    - nombre_archivo (str): Identificador con el que se va a almacenar el archivo descargado, en la ruta 'ruta_drive' pasada como argumento.
      Ejemplo: "REFERENCE-v0.csv"

  Return
  -------------
    - archivo_df (DataFrame): Objeto DataFrame que se ha creado a partir del archivo con extensión '.csv' descargado desde la ruta 'ruta_csv' de PhysioNet.

  Raises
  -------------
    - IsADirectoryError: La función muestra un mensaje de error si en lugar de especificar una ruta a un archivo (el que vamos a almacenar), esta corresponde a un directorio
    - ValueError: Se muestra un mensaje de error por pantalla si no se puede completar la descarga del archivo desde la ruta especificada.

    """

  from google.colab import drive                                   # Monta o se conecta a Google Drive para poder almacenar el archivo con extensión '.csv'
  drive.mount('/content/drive')

  os.makedirs(ruta_drive, exist_ok=True)                           # Comprueba si existe un directorio con el nombre 'ruta_drive', y si no es así, lo crea en Google Drive


  try:                                                             # Intenta realizar la descarga del archivo de clasificación desde Google Drive
    archivo_df = pd.read_csv(ruta_csv, sep = ',', header = None)   # Realiza la lectura del archivo con extensión '.csv'. Es como que accede a él de esta manera
    ruta_guardado = os.path.join(str(ruta_drive), str(nombre_archivo))   # Crea la ruta de guardado del archivo, incluyendo el nombre con el que se va a almacenar este

    if os.path.isdir(ruta_guardado):                               # Comprueba si la ruta es un directorio, y en caso afirmativo, muestra un error por pantalla
      raise IsADirectoryError(f"La ruta {ruta_guardado} es un directorio y se necesita un nombre de archivo para el almacenamiento del mismo. Inténtalo de nuevo.")

    archivo_csv = archivo_df.to_csv(ruta_guardado, index=False)                 # Si no es un directorio, realiza el guardado del archivo en la dirección 'ruta_guardado'

    if os.path.exists(ruta_guardado):                              # Comprueba si el archivo se ha guardado correctamente en la ruta especificada y muestra la confirmación
      print (f"El archivo .csv se ha guardado correctamente en la ruta pasada como argumento: {ruta_guardado}")

    return archivo_df

  except Exception as e:                                           # Si no es posible realizar la descarga del archivo, muestra un mensaje de error por pantalla
    raise ValueError (f" Ha ocurrido un error durante la descarga del archivo desde la ruta: {ruta_csv}. Inténtalo de nuevo")




In [ ]:
# Comprobamos que la función que hemos creado funciona correctamente

# Tenemos varios archivos .csv en PhysioNet, hay que decidir cual vamos a usar, aunque normalmente se usa el más reciente que es en el que se han corregido los posibles fallos


ruta_physionet = "https://physionet.org/files/challenge-2017/1.0.0/REFERENCE-v3.csv"

ruta_guardado = "/content/drive/MyDrive/dataset_ECG/challenge2017/clasificacion"



df_reference_v3 = descargar_archivo_clasificacion (ruta_physionet, ruta_guardado, "REFERENCE-v3.csv")

print (df_reference_v3.head())

Mounted at /content/drive
El archivo .csv se ha guardado correctamente en la ruta pasada como argumento: /content/drive/MyDrive/dataset_ECG/challenge2017/clasificacion/REFERENCE-v3.csv
        0  1
0  A00001  N
1  A00002  N
2  A00003  N
3  A00004  A
4  A00005  A


# Modificación del DataFrame para la clasificación

In [ ]:
def obtener_parametros_df (dataframe, nombres_columnas, verbose):

  """
  Esta función modifica los nombres de las columnas del DataFrame pasado como argumento. El nombre de cada una de estas columnas va a corresponder a cada uno de los elementos
  de la lista 'nombres_columnas' pasada como argumento. Por lo tanto, para que sea posible realizar la asignación de los nombres a las columnas correspondientes es necesario que
  la lista presente tantos elementos como columnas integren el DataFrame.
  Además, la función muestra información adicional por pantalla, cuando el parámetro booleano 'verbose' reciba el argumento True. En caso de que esto ocurra, la función
  imprimirá tanto el número de registros que contiene para cada valor de la segunda columna, como el total de registros que contiene el DataFrame.


  Parámetros
  -------------
    - dataframe (pd.DataFrame): DataFrame original con los valores de clasificación sobre el que vamos a trabajar.

    - nombres_columnas (list): Lista que contiene tantos elementos de tipo string como columnas presenta el DataFrame y contiene los futuros nombres de estas columnas.
      Ejemplo: ["id_señal", "etiqueta_clasificacion"]

    - verbose (bool, optional): Parámetro booleano que en caso de ser True muestra información adicional acerca del DataFrame por pantalla.

  Return
  -------------
    - dataframe_modificado (pd.DataFrame): DataFrame al que se le han modificado los nombres de las columnas, por los que aparecen en la lista 'nombres_columnas'


  Raises
  -------------
    - ValueError: Si la longitud de la lista 'nombres_columnas' no es igual al número de columnas del DataFrame, se muestra un mensaje de error por pantalla, porque no puede hacer
    la asignación de los nombres de las columnas correctamente.


  """

  if len(nombres_columnas) == dataframe.shape[1]:                          # Comprueba que la longitud de 'nombres_columnas' sea igual al número de columnas del DataFrame
    dataframe_modificado = dataframe.copy()                                # Realiza una copia del DataFrame original pasado como argumento
    dataframe_modificado.columns = nombres_columnas                        # Modifica los nombres de las columnas del DataFrame

  else:                                                                    # Si las longitudes no son iguales muestra un mensaje de error por pantalla
    raise ValueError(f"La longitud del parámetro 'nombres_columnas': {len(nombres_columnas)} debe ser igual al número de columnas del Dataframe: {dataframe.shape[1]}")


  if verbose:                                                              # Si el valor 'verbose' es True muestra información adicional por pantalla
    print (dataframe_modificado[nombres_columnas[1]].value_counts())       # Imprime el número de registros con cada valor único de la segunda columna del DataFrame
    print (len(dataframe_modificado))                                      # Imprime el número total de registros del DataFrame cuyos índices han sido modificados

  return dataframe_modificado                                              # Devuelve el DataFrame con los nombres de las columnas actualizados



In [ ]:
#Hacemos una prueba de que la función es correcta y funciona:

cols = ["ID", "Valor_señal"]

obtener_parametros_df (df_reference_v3, cols, True)

Valor_señal
N    5076
O    2415
A     758
~     279
Name: count, dtype: int64
8528


,ID,Valor_señal
0,A00001,N
1,A00002,N
2,A00003,N
3,A00004,A
4,A00005,A
...,...,...
8523,A08524,N
8524,A08525,O
8525,A08526,N
8526,A08527,N


In [ ]:
def obtener_archivo_valor (dataframe, nombres_columnas, valor_clasificacion, ruta_drive, verbose=False):

  """
  Esta función filtra el DataFrame pasado como argumento según el valor de clasificación pasado también como argumento, que equivale a una de las condiciones clínicas
  presentes en los registros del DataFrame y almacena este archivo resultante con extensión '.csv' en la ruta de Google Drive que se pasa como argumento.
  Además, si el parámetro booleano 'verbose' es True, muestra información adicional por pantalla acerca de los registros que contiene el DataFrame pasado como argumento.


  Parámetros
  -------------
    - dataframe (pd.DataFrame): DarataFrame original sobre el que se va a realizar el filtrado en función del valor de clasificación 'valor_clasificacion' pasado como argumento.

    - nombres_columnas (list): Lista con los nombres que se van a asignar a cada una de las columnas de las que se compone el DataFrame original.
      Ejemplo: ["id_señal", "etiqueta_clasificacion"]

    - valor_clasificacion (str): Valor de filtrado del DataFrame , se van a almacenar en un archivo '.csv' todos los registros que presenten dicho valor.
      Ejemplo: "N", "A", "O", "~"

    - ruta_drive (str): Ruta absoluta de Google Drive en la que se va a almacenar el archivo con extensión '.csv' con los registros que han sido filtrados.
      Ejemplo: "/content/drive/MyDrive/dataset_ECG/challenge2017/clasificacion"

    - verbose = False (bool, optional): Parámetro booleano que indica si se va a mostrar información adicional por pantalla del DataFrame en caso de que sea True. Es False por defecto.


  Return
  -------------
    - None: Esta función no devuelve nada, sino que realiza el filtrado y guarda el archivo '.csv' en la ruta pasada como argumento (ruta_drive).


  Raises
  ------------
    - ValueError: Muestra un mensaje de error por pantalla en el caso de que el argumento 'valor_clasificacion' no sea válido para el DataFrame pasado como argumento
    - ValueError: En el caso de que no se pueda completar la descarga del archivo en la ruta especificada, imprime un mensaje de error por pantalla.

  """

  dataframe = obtener_parametros_df (dataframe, nombres_columnas, verbose)               # Llama a la función 'obtener_parametros_df' para modificar el nombre de las columnas

  valores_validos = dataframe[nombres_columnas[1]].unique()                              # Crea un array de Numpy con los valores válidos para realizar el filtrado


  if valor_clasificacion not in valores_validos:                                         # Comprueba que el valor se encuentre dentro de los válidos, y sino muestra un error por pantalla
    raise ValueError (f"El valor de clasificación {valor_clasificacion} no está permitido. Inténtelo de nuevo con uno válido.")


  nombre_dataframe = "df_clasificacion_" + str(valor_clasificacion) + ".csv"             # Crea una variable con el nombre del archivo '.csv' en función del valor de filtrado

  dataframe = dataframe[dataframe[nombres_columnas[1]]==valor_clasificacion]             # Realiza la operación de filtrado por el 'valor_clasificacion' en sí misma


  try:                                                                                   # Intenta realizar el guardado del archivo que ha filtrado en la ruta especificada
    dataframe.to_csv(os.path.join(str(ruta_drive), str(nombre_dataframe)), index=False)  # Almacena el archivo con los valores filtrados en la ruta de Drive pasada como argumento

  except Exception as e:                                                                 # Si no es posible hacer la descarga del archivo, muestra un error por pantalla
    raise ValueError (f"No se ha podido completar la descarga del archivo con los valores filtrados. Inténtelo de nuevo. ")


In [ ]:
columnas = ["id", "clasificacion"]

In [ ]:
obtener_parametros_df (df_reference_v3, columnas, True)

clasificacion
N    5076
O    2415
A     758
~     279
Name: count, dtype: int64
8528


,id,clasificacion
0,A00001,N
1,A00002,N
2,A00003,N
3,A00004,A
4,A00005,A
...,...,...
8523,A08524,N
8524,A08525,O
8525,A08526,N
8526,A08527,N


In [ ]:
obtener_archivo_valor (df_reference_v3, columnas, "N", ruta_guardado)

In [ ]:
obtener_archivo_valor (df_reference_v3, columnas, "O", ruta_guardado)

In [ ]:
obtener_archivo_valor (df_reference_v3, columnas, "A", ruta_guardado)

In [ ]:
obtener_archivo_valor (df_reference_v3, columnas, "~", ruta_guardado)

In [ ]:
def filtrar_señales_valor (ruta_raiz_diccionarios, ruta_clasificacion, ruta_guardado):

  nuevo_dic = {}
  archivo_clasificacion = pd.read_csv (ruta_clasificacion)

  set_archivos_clasif = set(archivo_clasificacion["id"].unique())


  for i in os.listdir(ruta_raiz_diccionarios):
    ruta_carpeta = os.path.join(str(ruta_raiz_diccionarios), str(i))

    if os.path.isdir(ruta_carpeta):
      dic_señales = carga_archivos_entorno (ruta_carpeta)

      for e in dic_señales:
        if e in set_archivos_clasif:
          nuevo_dic[e] = dic_señales[e]

  with open (ruta_guardado, "wb") as archivo:
    pickle.dump(nuevo_dic, archivo)
    print ("Diccionario guardado correctamente en formato .pkl")

  return nuevo_dic




In [ ]:
dic = filtrar_señales_valor ("/content/drive/MyDrive/dataset_ECG/challenge2017/entrenamiento", "/content/drive/MyDrive/dataset_ECG/challenge2017/clasificacion/df_clasificacion_O.csv", "/content/drive/MyDrive/dataset_ECG/challenge2017/entrenamiento_GAN/señales_O")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/conte

In [ ]:
dic1 = filtrar_señales_valor ("/content/drive/MyDrive/dataset_ECG/challenge2017/entrenamiento", "/content/drive/MyDrive/dataset_ECG/challenge2017/clasificacion/df_clasificacion_A.csv", "/content/drive/MyDrive/dataset_ECG/challenge2017/entrenamiento_GAN/señales_A")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/conte

In [ ]:
dic2 = filtrar_señales_valor ("/content/drive/MyDrive/dataset_ECG/challenge2017/entrenamiento", "/content/drive/MyDrive/dataset_ECG/challenge2017/clasificacion/df_clasificacion_N.csv", "/content/drive/MyDrive/dataset_ECG/challenge2017/entrenamiento_GAN/señales_N")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/conte

In [ ]:
dic3 = filtrar_señales_valor ("/content/drive/MyDrive/dataset_ECG/challenge2017/entrenamiento", "/content/drive/MyDrive/dataset_ECG/challenge2017/clasificacion/df_clasificacion_~.csv", "/content/drive/MyDrive/dataset_ECG/challenge2017/entrenamiento_GAN/señales_~")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/conte

In [ ]:
print (len(dic))

2415


In [ ]:
print (len(dic1))

270


In [ ]:
print (len(dic2))

265


In [ ]:
print (len(dic3))

262


In [ ]:
print (len(dic4))

272


In [ ]:
print (len(dic5))

319


In [ ]:
print (len(dic6))

305


In [ ]:
print (len(dic7))

282


In [ ]:
print (len(dic8))

163
